# Week 1 · Day 2 — Lab 5
## Reproducible Randomness + End-to-End Capstone

Randomness is everywhere in AI work — weight init, dropout, shuffling, sampling,
synthetic data, train/test splits. If it isn't **reproducible**, your bugs aren't
either. NumPy 2.x's answer is the **`Generator`** API created by
`np.random.default_rng(seed)`. The legacy `np.random.seed()` / `np.random.rand()`
global functions still exist but are discouraged: they share one hidden global
state, which is exactly what makes "random" results impossible to reproduce.

This lab has two halves. First (~25 min) you drill the modern RNG: seeding,
distributions, sampling, shuffling, independent streams via `spawn`. Then
(~40 min) a **capstone** ties the whole day together — load-free synthetic eval
data, NaN-safe stats, normalization, masking, ranking, a cosine-similarity
retrieval step, and a saved results bundle.

### Learning objectives
1. Create seeded generators with `np.random.default_rng` and explain why global `np.random.*` is discouraged.
2. Draw from the common distributions (`random`, `standard_normal`, `uniform`, `integers`).
3. Sample without replacement (`choice`) and shuffle reproducibly.
4. Produce independent, non-overlapping streams with `Generator.spawn`.
5. **Capstone:** combine reductions, NaN-safe stats, broadcasting, masking, ranking, a little linear algebra, and `savez_compressed` into one analysis.

### Time budget — ~70 min
| Segment | Time |
|---|---|
| Framing & objectives | 4 min |
| **A.** Seeding & reproducibility | 8 min |
| **B.** Distributions | 7 min |
| **C.** Sampling & shuffling | 6 min |
| **D.** Independent streams (`spawn`) | 6 min |
| **CAP.** End-to-end eval pipeline | 36 min |
| Wrap-up | 3 min |

### Files you need
**None** — this lab generates everything from seeds. It *writes* one output
bundle (`eval_results.npz`) at the end.


In [ ]:
import numpy as np
from pathlib import Path

print("NumPy", np.__version__)   # target curriculum: NumPy 2.x on Python 3.13

# Solution is different here because of folder structure

OUT = Path("../data")
if not OUT.exists():
    OUT = Path(".")

def check(label, predicate):
    try:
        ok = bool(predicate())
    except Exception as exc:
        ok = False
        label = f"{label}  (raised {type(exc).__name__}: {exc})"
    print(("PASS " if ok else "FAIL "), label)
    return ok

# This lab GENERATES its own data with the modern Generator API — that is the
# lesson. There are no provided .npy inputs to load.
print("ready — this lab makes its own random data, reproducibly.")

## Part A — Seeding & reproducibility  *(guided)*

`np.random.default_rng(seed)` returns a `Generator`. Two generators built from
the **same seed** produce the **same stream** — that's reproducibility. The old
`np.random.seed(...)` + `np.random.rand(...)` pattern mutates one global state
shared across your whole process, so any other code that draws a random number
shifts your results. Prefer an explicit generator object you pass around.


In [ ]:
rng = np.random.default_rng(42)
print("three draws:", rng.random(3).round(4))

a = np.random.default_rng(7).random(3)
b = np.random.default_rng(7).random(3)
print("same seed -> identical stream:", np.array_equal(a, b))

### Exercise A1 — Prove reproducibility
Make two generators from seed `2025`, draw 5 floats from each into `draw_a` and
`draw_b`, and confirm they're identical. Then make a generator from a *different*
seed (`2026`) into `draw_c` and confirm it differs.


In [ ]:
draw_a = np.random.default_rng(2025).random(5)
draw_b = np.random.default_rng(2025).random(5)
draw_c = np.random.default_rng(2026).random(5)
print("a == b (same seed)  :", np.array_equal(draw_a, draw_b))
print("a == c (diff seed)  :", np.array_equal(draw_a, draw_c))

In [ ]:
check("A1: same seed -> identical streams", lambda: np.array_equal(draw_a, draw_b))
check("A1: different seed -> different stream", lambda: not np.array_equal(draw_a, draw_c))

🧑‍🏫 **Instructor note — A1.** The one-liner to make stick: *seed the generator,
not the globals.* Pass the `rng` object into functions explicitly. Mention that
`default_rng` uses PCG64 (better statistical quality than the legacy Mersenne
Twister) — but the headline reason to switch is reproducible, isolated state.


## Part B — Distributions

The `Generator` exposes the draws you'll use constantly:
- `rng.random(size)` — uniform in [0, 1),
- `rng.standard_normal(size)` — standard Gaussian (mean 0, std 1),
- `rng.uniform(low, high, size)` — uniform in [low, high),
- `rng.integers(low, high, size)` — random ints (high exclusive by default).


In [ ]:
rng = np.random.default_rng(0)
print("uniform[0,1):", rng.random(3).round(3))
print("gaussian    :", rng.standard_normal(3).round(3))
print("uniform 5-10:", rng.uniform(5, 10, 3).round(3))
print("ints 0-99   :", rng.integers(0, 100, 5))

### Exercise B1 — A synthetic batch
From one generator seeded `123`, draw:
- `latencies` — 1000 values uniform in [50, 250) (milliseconds),
- `noise` — 1000 standard-normal values,
- `token_counts` — 1000 integers in [10, 512).

Then report the mean latency (should land near 150).


In [ ]:
rng = np.random.default_rng(123)
latencies = rng.uniform(50, 250, 1000)
noise = rng.standard_normal(1000)
token_counts = rng.integers(10, 512, 1000)
print("mean latency:", latencies.mean().round(2))
print("noise mean ~0:", noise.mean().round(3), "| std ~1:", noise.std().round(3))
print("token range:", token_counts.min(), "->", token_counts.max())

In [ ]:
check("B1: latencies in [50, 250)",
      lambda: float(latencies.min()) >= 50 and float(latencies.max()) < 250)
check("B1: token_counts in [10, 512)",
      lambda: int(token_counts.min()) >= 10 and int(token_counts.max()) < 512)
check("B1: noise looks standard-normal",
      lambda: abs(float(noise.mean())) < 0.1 and abs(float(noise.std()) - 1) < 0.1)

## Part C — Sampling & shuffling

- `rng.choice(a, size, replace=False)` — sample **without** replacement (a random
  subset; no element repeats).
- `rng.permutation(n)` — a fresh shuffled `arange(n)` (returns a new array).
- `rng.shuffle(arr)` — shuffle an array **in place**.


In [ ]:
rng = np.random.default_rng(1)
pool = np.arange(10)
print("subset of 4 (no repeats):", rng.choice(pool, size=4, replace=False))
print("permutation of 0..9     :", rng.permutation(10))

### Exercise C1 — A reproducible train/test split
You have 200 example indices. Using a generator seeded `99`, draw a
**without-replacement** sample of 40 indices as `test_idx`, then make `train_idx`
the remaining 160 (use `np.setdiff1d`). Confirm they don't overlap and cover all 200.


In [ ]:
rng = np.random.default_rng(99)
all_idx = np.arange(200)
test_idx = rng.choice(200, size=40, replace=False)
train_idx = np.setdiff1d(all_idx, test_idx)
print("test:", test_idx.shape[0], "| train:", train_idx.shape[0])
print("overlap:", np.intersect1d(test_idx, train_idx).size)

In [ ]:
check("C1: 40 test + 160 train",
      lambda: test_idx.shape[0] == 40 and train_idx.shape[0] == 160)
check("C1: no overlap between splits",
      lambda: np.intersect1d(test_idx, train_idx).size == 0)
check("C1: splits cover all 200 indices",
      lambda: np.array_equal(np.union1d(test_idx, train_idx), np.arange(200)))

🧑‍🏫 **Instructor note — C1.** `replace=False` is what guarantees a clean split —
a frequent bug is sampling *with* replacement and getting duplicate test rows.
`np.setdiff1d` returns sorted unique values, which is exactly what we want for the
complement. Same-seed → same split every run, which is how you make experiments
comparable.


## Part D — Independent streams with `spawn`

Need several generators that are guaranteed **not** to overlap (parallel workers,
per-fold splits, multiple augmentation pipelines)? Don't seed them with
`seed`, `seed+1`, `seed+2` — those streams can correlate. Use **`spawn`**, which
derives statistically independent child generators from a parent's seed sequence.


In [ ]:
parent = np.random.default_rng(2024)
kids = parent.spawn(3)                       # 3 independent child Generators
for i, k in enumerate(kids):
    print(f"child {i}:", k.random(3).round(4))

### Exercise D1 — Four independent workers, reproducibly
From a parent seeded `7`, spawn **4** children. Collect each child's 5 draws into
a list `streams` (a list of 4 arrays). Then prove reproducibility: respawn from a
*fresh* parent seeded `7`, take child 0's 5 draws as `replay`, and confirm it
equals `streams[0]`.


In [ ]:
parent = np.random.default_rng(7)
children = parent.spawn(4)
streams = [child.random(5) for child in children]

replay = np.random.default_rng(7).spawn(4)[0].random(5)
print("num streams:", len(streams))
print("child0 reproduced:", np.array_equal(streams[0], replay))

In [ ]:
check("D1: four independent streams", lambda: len(streams) == 4)
check("D1: each stream has 5 draws",
      lambda: all(s.shape == (5,) for s in streams))
check("D1: spawn is reproducible from the same parent seed",
      lambda: np.array_equal(streams[0], replay))
check("D1: distinct children differ",
      lambda: not np.array_equal(streams[0], streams[1]))

🧑‍🏫 **Instructor note — D1.** `spawn` (added in the modern API) is the *correct*
way to seed parallel work; the anti-pattern is `default_rng(base+i)`. Emphasize
that spawning is deterministic from the parent — same parent seed → same set of
children, so even your "parallel" randomness is reproducible. This matters the
moment they touch multiprocessing or sharded data loaders.


## CAPSTONE — End-to-end eval-score analysis  *(~36 min)*

This pulls the **entire day** together into one realistic task. You are handed a
batch of model responses, each with a few quality metrics (some missing) and an
embedding vector. Your job: clean, normalize, rank, retrieve, and persist.

You'll generate the data yourself (reproducibly), so the whole pipeline is
runnable anywhere with no external files. Work through the steps in order — each
builds on the previous one.


### CAP-0 — Generate the synthetic batch  *(guided — run as given)*
200 responses × 3 metrics, plus a 16-dim embedding per response. We inject ~5%
missing values into the metrics to exercise the NaN-safe path.


In [ ]:
rng = np.random.default_rng(42)
N, M, D = 200, 3, 16
METRICS = np.array(["relevance", "coherence", "safety"])

# Quality metrics in [0, 1], then knock out ~5% of cells as NaN.
eval_metrics = rng.uniform(0.3, 1.0, size=(N, M))
mask_missing = rng.random((N, M)) < 0.05
eval_metrics[mask_missing] = np.nan

# A 16-dim embedding per response (think: sentence embedding).
embeddings = rng.standard_normal((N, D))

print("eval_metrics:", eval_metrics.shape, "| missing cells:", int(np.isnan(eval_metrics).sum()))
print("embeddings  :", embeddings.shape)

### CAP-1 — Trustworthy column stats
Compute NaN-safe per-metric `means` and `stds` (shape `(3,)` each, no NaN), and
`missing_per_metric` (shape `(3,)`).


In [ ]:
means = np.nanmean(eval_metrics, axis=0)
stds = np.nanstd(eval_metrics, axis=0)
missing_per_metric = np.isnan(eval_metrics).sum(axis=0)
for name, mu, sd, miss in zip(METRICS, means, stds, missing_per_metric):
    print(f"{name:10s} mean={mu:.3f} std={sd:.3f} missing={int(miss)}")

In [ ]:
check("CAP-1: means shape (3,), no NaN",
      lambda: means.shape == (3,) and not np.isnan(means).any())
check("CAP-1: stds shape (3,), no NaN",
      lambda: stds.shape == (3,) and not np.isnan(stds).any())
check("CAP-1: missing counts match the matrix",
      lambda: int(missing_per_metric.sum()) == int(np.isnan(eval_metrics).sum()))

### CAP-2 — Impute, then z-score normalize
Replace missing cells with that metric's mean (mean-imputation via `np.where`),
giving `filled`. Then z-score normalize with **keepdims** broadcasting:
`normed = (filled - means_kd) / stds_kd`. Verify each column of `normed` has mean
~0 and std ~1.


In [ ]:
filled = np.where(np.isnan(eval_metrics), means, eval_metrics)
means_kd = filled.mean(axis=0, keepdims=True)
stds_kd = filled.std(axis=0, keepdims=True)
normed = (filled - means_kd) / stds_kd
print("normed col means (~0):", normed.mean(axis=0).round(12))
print("normed col stds  (~1):", normed.std(axis=0).round(6))

In [ ]:
check("CAP-2: filled has no NaN", lambda: not np.isnan(filled).any())
check("CAP-2: normed columns mean ~0",
      lambda: bool(np.allclose(normed.mean(axis=0), 0.0, atol=1e-9)))
check("CAP-2: normed columns std ~1",
      lambda: bool(np.allclose(normed.std(axis=0), 1.0, atol=1e-6)))

### CAP-3 — Composite score, flagging & ranking
Build a per-response `composite` = mean across the 3 **filled** metrics. Flag the
weak responses: those whose composite is **below the median**. Then rank everyone
descending and take the `top10` response indices.


In [ ]:
composite = filled.mean(axis=1)
threshold = np.median(composite)
weak_mask = composite < threshold
weak_idx = np.where(weak_mask)[0]
top10 = np.argsort(composite)[-10:][::-1]
print("median composite:", round(float(threshold), 4))
print("weak responses  :", weak_idx.size)
print("top10 indices   :", top10)

In [ ]:
check("CAP-3: composite shape (200,)", lambda: composite.shape == (200,))
check("CAP-3: ~half are below the median",
      lambda: abs(weak_idx.size - 100) <= 1)
check("CAP-3: top10 sorted descending",
      lambda: bool(np.all(np.diff(composite[top10]) <= 0)))
check("CAP-3: best response is top10[0]",
      lambda: int(top10[0]) == int(np.argmax(composite)))

🧑‍🏫 **Instructor note — CAP-3.** "Below the median" → roughly 100 flagged (exact
count can be 99–101 depending on ties at the median). This is the day's core loop
in miniature: reduce → threshold → mask → rank. Connect it to real eval triage:
auto-flag low-scoring generations for human review.


### CAP-4 — Cosine-similarity retrieval  *(the AI payoff)*
Embeddings let you ask "which responses are most **similar**?" Cosine similarity
is the dot product of **L2-normalized** vectors. Normalize every embedding to unit
length, then the full similarity matrix is just `unit @ unit.T`.

Use the best response (`query = top10[0]`) as the query and retrieve its 5 nearest
neighbors (excluding itself).


In [ ]:
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
unit = embeddings / norms
sim = unit @ unit.T
query = int(top10[0])

row = sim[query].copy()
row[query] = -np.inf                       # exclude self
neighbors = np.argsort(row)[-5:][::-1]
print("query response:", query)
print("nearest 5      :", neighbors)
print("their cos-sim  :", sim[query, neighbors].round(4))

In [ ]:
check("CAP-4: every embedding normalized to unit length",
      lambda: bool(np.allclose(np.linalg.norm(unit, axis=1), 1.0)))
check("CAP-4: sim diagonal is ~1 (self-similarity)",
      lambda: bool(np.allclose(np.diag(sim), 1.0)))
check("CAP-4: 5 neighbors, none is the query itself",
      lambda: neighbors.size == 5 and query not in neighbors.tolist())
check("CAP-4: neighbors sorted by descending similarity",
      lambda: bool(np.all(np.diff(sim[query, neighbors]) <= 0)))

🧑‍🏫 **Instructor note — CAP-4.** This is the conceptual core of semantic search /
RAG retrieval, built from two NumPy primitives: L2-normalize, then a matrix
multiply. Hammer the distinction: `@` / `np.matmul` is matrix multiply (the
similarity matrix); `*` is elementwise (NOT what you want here). The
self-exclusion trick (`-inf` on the diagonal entry) is a standard retrieval
detail. Diagonal ~1.0 because a unit vector dotted with itself is 1.


### CAP-5 — Persist the results bundle
Save the analysis to a compressed `.npz` so it round-trips. Bundle `composite`,
`top10`, `weak_idx`, and `sim`, then reload and confirm `top10` survived intact.


In [ ]:
out_path = OUT / "eval_results.npz"
np.savez_compressed(out_path, composite=composite, top10=top10,
                    weak_idx=weak_idx, sim=sim)
loaded = np.load(out_path)
top10_roundtrip = loaded["top10"]
print("saved:", out_path.name, "| keys:", list(loaded.keys()))
print("top10 survived round-trip:", np.array_equal(top10_roundtrip, top10))

In [ ]:
check("CAP-5: bundle has all four arrays",
      lambda: set(loaded.keys()) == {"composite", "top10", "weak_idx", "sim"})
check("CAP-5: top10 round-trips exactly",
      lambda: np.array_equal(top10_roundtrip, top10))

🧑‍🏫 **Instructor note — CAP-5.** `savez_compressed` keeps named arrays in one
file; `np.load` returns a lazy dict-like `NpzFile` keyed by the names you passed.
This is the lightweight way to checkpoint analysis artifacts between pipeline
stages before they reach for heavier formats. The file lands in `data/` (or the
notebook dir).


## Wrap-up — what you can now do

- Create seeded `Generator`s with `np.random.default_rng` and explain why the global `np.random.*` API is discouraged.
- Draw from uniform, Gaussian, and integer distributions, and sample/shuffle reproducibly.
- Produce independent, reproducible parallel streams with `spawn`.
- Run a full eval pipeline end-to-end: NaN-safe stats → impute → z-score normalize → composite → flag → rank → cosine-similarity retrieve → save.

**That's Day 2.** You can now reason about an ndarray's shape, dtype, and memory;
vectorize and broadcast instead of looping; index with views/masks/fancy
selection; reduce, sort, and rank with NaN-safe stats; and generate reproducible
randomness — the NumPy foundation the rest of the Academy builds on.
